# The Impact of Private Use Terminals (TUPs) on Formal Employment in Municipalities: A Synthetic Control Approach
Bruno Duarte.

**Outcome:** formal employment stock at 31/12 (RAIS, all economic activities summed), by municipality and year.
**Specification:** lags-only predictors (all pre-treatment lags of the outcome), as in the GDP per capita model.
**Scope of this notebook:** main model and plots only; robustness checks are left for a later step.

## 0. Libraries

In [ ]:
#Import the necessary libraries

from pathlib import Path
import pandas as pd
import pysyncon
from pysyncon import Dataprep, Synth
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

## 1. Parameters and output folder
Choose the treated municipality in `treated_mun`; everything else follows from the `TREATED` dictionary.

In [ ]:
#Set the parameters for the treated municipality

TREATED = {
    'Itapoá':     {'code': 4208450, 'year': 2011},   # IBGE code, year the port facility began operating
    'Navegantes': {'code': 4211306, 'year': 2007},   # CONFIRM the treatment year before running
}

treated_mun = 'Itapoá'          # 'Itapoá' or 'Navegantes'
treated_code = TREATED[treated_mun]['code']
treatment_year = TREATED[treated_mun]['year']
treatment_state = 'SC'          # state (UF) used to filter the donor pool

first_year = 1995               # first year of the panel (see section 2.4)
last_year = 2025                # last year of the panel

In [ ]:
#Paths: RAIS panel, IBGE code/name table, and the original database (used only for the IP flag)

path_rais = r'D:\Users\Bruno\Documents\Dissertacao\3. First Essay - TUPs impact\05. Python\data\processed\painel_rais_mun_total.parquet'
path_mun = r'D:\Users\Bruno\Documents\Dissertacao\3. First Essay - TUPs impact\05. Python\mun_ibge.xlsx'
path_bd = r'D:\Users\Bruno\Documents\Dissertacao\7. Papers\1.TUPs impact\02. Dados\bd_BA_ES_SC.xlsx'

#Define the output folder for the results

results_path = Path(r"D:\Users\Bruno\Documents\Dissertacao\7. Papers\1.TUPs impact\04. Python\Resultados") / 'Employment stock' / treated_mun
results_path.mkdir(parents=True, exist_ok=True)

## 2. Data loading and filtering

### 2.1 RAIS employment stock
The RAIS panel has one row per municipality and year (`ano`, `uf`, `mun`, `estoque`). `mun` is the 6-digit municipality code, stored as text.

In [ ]:
#Load the RAIS panel (total formal employment stock at 31/12) and keep the treated state

rais = pd.read_parquet(path_rais)
rais = rais[rais['uf'] == treatment_state].rename(columns={'mun': 'mun6'}).copy()
print(f"RAIS rows ({treatment_state}): {len(rais)} | municipalities: {rais['mun6'].nunique()} | years: {rais['ano'].min()}-{rais['ano'].max()}")

### 2.2 Municipality codes
RAIS codes have 6 digits (IBGE code without the check digit). The correspondence table gives the 7-digit IBGE code and the name; the match is made on the first 6 digits.

In [ ]:
#Link the 6-digit RAIS code to the 7-digit IBGE code and the municipality name

mun_ibge = pd.read_excel(path_mun, dtype=str).drop_duplicates()
mun_ibge['mun6'] = mun_ibge['cod_mun'].str[:6]
mun_ibge['cod_mun'] = mun_ibge['cod_mun'].astype(int)

df = rais.merge(mun_ibge, on='mun6', how='inner')

only_rais = sorted(set(rais['mun6']) - set(mun_ibge['mun6']))
only_table = sorted(set(mun_ibge['mun6']) - set(rais['mun6']))
print(f"RAIS codes without a match in the IBGE table ({len(only_rais)}): {only_rais}")
print(f"IBGE table codes absent from RAIS ({len(only_table)}): {only_table}")
print(f"Municipalities after the match: {df['cod_mun'].nunique()}")

### 2.3 Donor-pool flag (IP)
The donor pool keeps only municipalities flagged `não` in the `IP` column of the original database, plus the treated one. The employment panel does not carry this flag, so it is read from that database.

In [ ]:
#Read the IP flag from the original database and attach it to the RAIS panel

attrs = pd.read_excel(path_bd, sheet_name='data', header=0, usecols=['cod_mun', 'IP'])
attrs['cod_mun'] = attrs['cod_mun'].astype(int)

if (attrs.groupby('cod_mun')['IP'].nunique() > 1).any():
    print("WARNING: IP varies over time for some municipalities; the first value is used.")
attrs = attrs.drop_duplicates('cod_mun')

df = df.merge(attrs, on='cod_mun', how='left')
df.loc[df['cod_mun'] == treated_code, 'IP'] = 'tratado'

print("IP values (one row per municipality):")
print(df.drop_duplicates('cod_mun')['IP'].value_counts(dropna=False))

### 2.4 Panel window
Zeros in the RAIS panel can mean that a municipality did not exist yet. Such units distort the fit, so donors with zero employment in any pre-treatment year are dropped. The table below shows how many donors remain for different start years, to help choose `first_year`.

In [ ]:
#How many donors remain for each possible first year of the panel

df_flag = df[df['IP'].isin(['não', 'tratado'])]

rows = []
for y0 in [1985, 1990, 1995, 2000, 2005]:
    pre_ = df_flag[(df_flag['ano'] >= y0) & (df_flag['ano'] < treatment_year)]
    zero_units_ = set(pre_.loc[pre_['estoque'] == 0, 'cod_mun'])
    donors_ = [m for m in pre_['cod_mun'].unique() if m != treated_code and m not in zero_units_]
    rows.append({'first_year': y0, 'pre_treatment_years': treatment_year - y0,
                 'treated_has_zeros': treated_code in zero_units_, 'donors_without_zeros': len(donors_)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
#Build the final panel and the donor pool

df_state = df[df['IP'].isin(['não', 'tratado']) & df['ano'].between(first_year, last_year)].copy()

pre = df_state[df_state['ano'] < treatment_year]
zero_units = set(pre.loc[pre['estoque'] == 0, 'cod_mun'])

assert treated_code not in zero_units, (
    f"{treated_mun} has zero employment in some pre-treatment year; increase first_year."
)

control_mun = [m for m in df_state['cod_mun'].unique() if m != treated_code and m not in zero_units]
df_state = df_state[df_state['cod_mun'].isin(control_mun + [treated_code])].copy()

pre_years = range(first_year, treatment_year)
all_years = range(first_year, last_year + 1)

n_years = len(all_years)
assert (df_state.groupby('cod_mun')['ano'].nunique() == n_years).all(), "Unbalanced panel."

print(f"Treated: {treated_mun} ({treated_code}), treatment year {treatment_year}")
print(f"Panel: {first_year}-{last_year} | pre-treatment years: {len(pre_years)}")
print(f"Donors dropped for zeros in the pre-period: {len(zero_units)}")
print(f"Donor pool: {len(control_mun)} municipalities")

## 3. Main Synthetic Control Model
Predictors are lags-only: all pre-treatment lags of the employment stock are passed as `special_predictors`.

In [ ]:
#Prepare the model using the Dataprep class from the pysyncon library

dataprep = Dataprep(
    foo=df_state,
    predictors=[],
    predictors_op='mean',
    time_predictors_prior=pre_years,
    special_predictors=[
        ('estoque', [year], 'mean') for year in pre_years
    ],
    dependent='estoque',
    unit_variable='cod_mun',
    time_variable='ano',
    treatment_identifier=treated_code,
    controls_identifier=control_mun,
    time_optimize_ssr=pre_years,
)

In [ ]:
#Fit the model using the Synth class from the pysyncon library

synth = Synth()
synth.fit(dataprep, optim_method='L-BFGS-B', optim_initial='ols')

att_result = synth.att(time_period=range(treatment_year, last_year + 1))
print(f"ATT ({treated_mun}):")
print(att_result)

In [ ]:
#Get model variables and weights

synth.summary()

In [ ]:
#Build a DataFrame with the control municipalities' weights and export it

synth_weights_exact = pd.Series(
    synth.W, index=list(dataprep.controls_identifier), name='weights'
)

WEIGHT_THRESHOLD = 0.0001

df_weights = synth_weights_exact.reset_index()
df_weights.columns = ['cod_mun', 'weight']
df_weights = df_weights[df_weights['weight'] > WEIGHT_THRESHOLD].reset_index(drop=True)
df_weights = df_weights.merge(df[['cod_mun', 'mun']].drop_duplicates(), on='cod_mun', how='left')
df_weights = df_weights.sort_values('weight', ascending=False).reset_index(drop=True)

print(df_weights)
df_weights.to_excel(results_path / f"{treated_mun}_weights_controlmun.xlsx", index=False)

In [ ]:
#Compare the treated municipality's employment stock with the synthetic employment stock

actual_emp = (
    df_state[df_state['cod_mun'] == treated_code]
    .set_index('ano')['estoque']
    .reindex(all_years)
)

control_pivot = (
    df_state[df_state['cod_mun'].isin(control_mun)]
    .pivot(index='ano', columns='cod_mun', values='estoque')
    .reindex(all_years)
)

synthetic_emp = control_pivot[synth_weights_exact.index].mul(synth_weights_exact.values, axis=1).sum(axis=1)

df_results = pd.DataFrame({
    'ano': list(all_years),
    'real_employment': actual_emp.values,
    'synthetic_employment': synthetic_emp.values,
})

df_results['difference'] = df_results['real_employment'] - df_results['synthetic_employment']
df_results['period'] = df_results['ano'].apply(
    lambda y: 'before treatment' if y < treatment_year else 'after treatment'
)

print(df_results)

#Average Treatment Effect on the Treated (ATT) after treatment
avg_effect_post = df_results.loc[
    df_results['ano'] >= treatment_year, 'difference'
].mean()
print(f"Average Treatment Effect on the Treated (ATT) after treatment: {avg_effect_post:.2f}")

#Average difference between the treated municipality and the synthetic control before treatment
avg_effect_pre = df_results.loc[
    df_results['ano'] < treatment_year, 'difference'
].mean()
print(f"Average difference before treatment: {avg_effect_pre:.2f}")

#Root Mean Squared Prediction Error (RMSPE) before and after treatment
def rmspe(diffs):
    return (diffs ** 2).mean() ** 0.5

rmspe_pre = rmspe(df_results.loc[df_results['ano'] < treatment_year, 'difference'])
print(f"Pre-treatment RMSPE: {rmspe_pre:.2f}")

rmspe_post = rmspe(df_results.loc[df_results['ano'] >= treatment_year, 'difference'])
print(f"Post-treatment RMSPE: {rmspe_post:.2f}")

rmspe_ratio = rmspe_post / rmspe_pre
print(f"Post/pre RMSPE ratio: {rmspe_ratio:.2f}")

#Build a summary DataFrame with the main results
df_effects = pd.DataFrame({
    'mun': [treated_mun],
    'treatment_year': [treatment_year],
    'n_donors': [len(control_mun)],
    'avg_effect_pre': [avg_effect_pre],
    'avg_effect_post': [avg_effect_post],
    'rmspe_pre': [rmspe_pre],
    'rmspe_post': [rmspe_post],
    'rmspe_ratio': [rmspe_ratio],
})

#Export the results to Excel files
df_results.to_excel(results_path / f"{treated_mun}_results.xlsx", index=False)
df_effects.to_excel(results_path / f"{treated_mun}_effects.xlsx", index=False)

## 4. Plots: real vs. synthetic, and the gap

In [ ]:
#Generate the styled plots

# --- Ensure years are integers in the DataFrame ---
df_results['ano'] = df_results['ano'].astype(int)

# --- Global style settings ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['text.color'] = '#333333'
plt.rcParams['axes.labelcolor'] = '#333333'
plt.rcParams['xtick.color'] = '#333333'
plt.rcParams['ytick.color'] = '#333333'

# --- Color palette ---
COLOR_ACTUAL = '#003F88'
COLOR_SYNTHETIC = '#3FA34D'
COLOR_TREATMENT = '#F77F00'
COLOR_GRID = '#ECECEC'

# ==========================================
# CHART 1: REAL VS SYNTHETIC
# ==========================================
fig1, ax1 = plt.subplots(figsize=(10, 5))
fig1.patch.set_alpha(0)
ax1.patch.set_alpha(0)

ax1.plot(df_results['ano'], df_results['real_employment'],
         color=COLOR_ACTUAL, linewidth=2.5, label=f'Real {treated_mun}')
ax1.plot(df_results['ano'], df_results['synthetic_employment'],
         color=COLOR_SYNTHETIC, linewidth=2.2, linestyle='--', label=f'Synthetic {treated_mun}')
ax1.axvline(treatment_year, color=COLOR_TREATMENT, linewidth=2.8, linestyle=':',
            label=f'Treatment ({treatment_year})')

ax1.set_ylabel('Formal employment stock at 31/12', fontsize=11, labelpad=10)
ax1.set_xlabel('Year', fontsize=11, labelpad=10)
ax1.set_ylim(bottom=0)

ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.spines['left'].set_color('#cccccc')
ax1.spines['bottom'].set_color('#cccccc')
ax1.grid(True, which='both', linestyle='-', linewidth=0.5, color=COLOR_GRID)
ax1.grid(False, axis='x')

ax1.legend(frameon=True, facecolor='white', edgecolor='none', fontsize=9, loc='upper left')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f"{int(x):,}".replace(",", ".")))

ax1.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x)}'))
plt.xticks(rotation=45, ha='right', fontsize=9)

plt.tight_layout()
plt.savefig(results_path / f"{treated_mun}_synthxreal.png", dpi=300, bbox_inches='tight', transparent=True)
plt.show()

# ==========================================
# CHART 2: GAP (DIFFERENCE)
# ==========================================
fig2, ax2 = plt.subplots(figsize=(10, 4.5))
fig2.patch.set_alpha(0)
ax2.patch.set_alpha(0)

ax2.plot(df_results['ano'], df_results['difference'],
         color=COLOR_ACTUAL, linewidth=2.5, label=f'Gap in formal employment for {treated_mun}')
ax2.axhline(0, color='#888888', linewidth=1.2, linestyle='--')
ax2.axvline(treatment_year, color=COLOR_TREATMENT, linewidth=2.8, linestyle=':',
            label=f'Treatment ({treatment_year})')

ax2.set_ylabel('Formal employment (real - synthetic)', fontsize=11, labelpad=10)
ax2.set_xlabel('Year', fontsize=11, labelpad=10)

ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['left'].set_color('#cccccc')
ax2.spines['bottom'].set_color('#cccccc')
ax2.grid(True, which='both', linestyle='-', linewidth=0.5, color=COLOR_GRID)
ax2.grid(False, axis='x')

ax2.legend(frameon=True, facecolor='white', edgecolor='none', fontsize=9, loc='upper left')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f"{int(x):,}".replace(",", ".")))

ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x)}'))
plt.xticks(rotation=45, ha='right', fontsize=9)

plt.tight_layout()
plt.savefig(results_path / f"{treated_mun}_employment_gap.png", dpi=300, bbox_inches='tight', transparent=True)
plt.show()